# Training Baseline & Benchmark Unicode — Deteksi Promosi Judol

Notebook mandiri untuk Google Colab atau Kaggle. Notebook ini melatih dua baseline pada data teks mentah: **TF-IDF character n-gram + Logistic Regression** dan **fine-tuning IndoBERT end-to-end**, lalu mengukur ketahanan checkpoint baseline yang *tidak diubah* terhadap manipulasi Unicode pada test split yang sudah dibekukan.

## Kontrak eksperimen

- Workbook: `dataset_sampled_2384_labeled.xlsx`, sheet `dataset_sampled_2384`.
- Input model **hanya**: `text_raw`.
- Ground truth **hanya**: kolom `label`.
- Kelas yang dipakai: `bukan_judol → 0`, `promosi_judol → 1`. Baris `perlu_review` dan `not_indonesian` dikeluarkan.
- Kolom metadata seperti `weak_label`, `sample_type`, `source_type`, `platform`, `matched_keywords`, URL, dan `text_normalized` **tidak pernah** dipakai sebagai fitur.
- Bagian baseline clean tidak melakukan normalisasi Unicode atau augmentasi pada `text_raw`.
- Split dibuat sekali dengan seed 42, bersifat group-aware untuk teks/template yang mirip, lalu disimpan sebagai artefak beku.
- Bagian benchmark Unicode hanya membuat varian dari **test split beku**, memuat ulang checkpoint baseline, dan tidak melatih ulang model. Threshold prediksi tetap `0.5`.

> Jika baseline sudah selesai dilatih, salin folder artefaknya ke Drive/Kaggle, jalankan sel setup/konfigurasi/fungsi evaluasi, lewati sel training, lalu jalankan bagian benchmark Unicode di akhir notebook.

## 1. Instal dependensi

Jalankan sel ini sekali setiap runtime baru. Aktifkan GPU pada Colab/Kaggle sebelum menjalankan bagian Transformer.

In [ ]:
# Colab dan Kaggle sudah menyediakan PyTorch + Transformers. Jangan upgrade `transformers`
# atau `huggingface-hub` di runtime bawaan karena dapat merusak dependency `diffusers`/`gradio`.
%pip install -q 'openpyxl>=3.1.5' 'scikit-learn>=1.5' 'datasets>=3.0' 'accelerate>=1.0' 'evaluate>=0.4' 'joblib>=1.4' seaborn


## 2. Konfigurasi path dan eksperimen

Ubah `DATASET_PATH` sesuai lokasi file Anda. Untuk Colab, upload workbook ke `/content/` atau gunakan Google Drive. Untuk Kaggle, pilih path di bawah `/kaggle/input/`.

In [ ]:
from pathlib import Path

# Contoh Colab upload: DATASET_PATH = Path('/content/dataset_sampled_2384_labeled.xlsx')
# Contoh Colab Drive : DATASET_PATH = Path('/content/drive/MyDrive/judol_detection/data/annotation/dataset_sampled_2384_labeled.xlsx')
# Contoh Kaggle      : DATASET_PATH = Path('/kaggle/input/NAMA-DATASET/dataset_sampled_2384_labeled.xlsx')
DATASET_PATH = Path('/content/dataset_sampled_2384_labeled.xlsx')  # <-- UBAH BILA PERLU
OUTPUT_DIR = Path('/content/judol_baseline_outputs')               # Kaggle: Path('/kaggle/working/judol_baseline_outputs')
SHEET_NAME = 'dataset_sampled_2384'

RNG_SEED = 42
TRAIN_FOLDS = 5       # 5/7 = 71.4%
VALIDATION_FOLDS = 1  # 1/7 = 14.3%
TEST_FOLDS = 1        # 1/7 = 14.3%

# Transformer baseline (turunkan batch size bila GPU kehabisan memori)
MODEL_NAME = 'indobenchmark/indobert-base-p1'
MODEL_REVISION = 'c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2'
PRETRAINED_MODEL_DIR = OUTPUT_DIR / 'pretrained_indobert'
MAX_LENGTH = 256
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_FOLDS + VALIDATION_FOLDS + TEST_FOLDS == 7
print('Dataset :', DATASET_PATH)
print('Output  :', OUTPUT_DIR.resolve())


In [ ]:
import hashlib
import json
import os
import random
import re
import sys
import unicodedata
from importlib.metadata import version

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    average_precision_score, precision_recall_fscore_support, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments,
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RNG_SEED)
pd.set_option('display.max_colwidth', 160)

runtime_info = {
    'python': sys.version,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'pandas': version('pandas'),
    'scikit_learn': version('scikit-learn'),
    'transformers': version('transformers'),
    'datasets': version('datasets'),
}
(OUTPUT_DIR / 'runtime_versions.json').write_text(json.dumps(runtime_info, indent=2), encoding='utf-8')
print(json.dumps(runtime_info, indent=2))

## 2.1 Download checkpoint IndoBERT yang dipin

Tokenizer dan bobot dasar diunduh sekali dari revision immutable yang sama dengan
provenance artifact backend, lalu disimpan di dalam `OUTPUT_DIR`. Sel training hanya
memuat folder lokal ini; tidak ada request model remote saat fine-tuning dimulai.

In [ ]:
if PRETRAINED_MODEL_DIR.exists() and any(PRETRAINED_MODEL_DIR.iterdir()):
    print('Menggunakan checkpoint lokal yang sudah ada:', PRETRAINED_MODEL_DIR)
else:
    PRETRAINED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    download_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    download_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        num_labels=2,
    )
    download_tokenizer.save_pretrained(PRETRAINED_MODEL_DIR)
    download_model.save_pretrained(PRETRAINED_MODEL_DIR)

required_pretrained_files = {'config.json', 'tokenizer_config.json'}
missing_pretrained_files = sorted(
    filename for filename in required_pretrained_files
    if not (PRETRAINED_MODEL_DIR / filename).is_file()
)
if not any((PRETRAINED_MODEL_DIR / filename).is_file() for filename in ('model.safetensors', 'pytorch_model.bin')):
    missing_pretrained_files.append('model.safetensors atau pytorch_model.bin')
if missing_pretrained_files:
    raise FileNotFoundError(
        'Checkpoint IndoBERT lokal tidak lengkap: ' + ', '.join(missing_pretrained_files)
    )

pretrained_model_metadata = {
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'local_path': str(PRETRAINED_MODEL_DIR.resolve()),
}
(OUTPUT_DIR / 'pretrained_indobert_metadata.json').write_text(
    json.dumps(pretrained_model_metadata, indent=2), encoding='utf-8'
)
print(json.dumps(pretrained_model_metadata, indent=2))

## 3. Load dan validasi data

Sel ini hanya mempertahankan `text_raw`, `label`, dan ID baris internal untuk artefak split. Metadata koleksi tidak diteruskan ke model.

In [ ]:
LABEL_MAP = {'bukan_judol': 0, 'promosi_judol': 1}
REQUIRED_COLUMNS = {'text_raw', 'label'}

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'Workbook tidak ditemukan: {DATASET_PATH}. Upload file atau ubah DATASET_PATH pada sel konfigurasi.'
    )

raw_df = pd.read_excel(DATASET_PATH, sheet_name=SHEET_NAME)
missing_columns = REQUIRED_COLUMNS - set(raw_df.columns)
if missing_columns:
    raise ValueError(f'Kolom wajib tidak ditemukan: {sorted(missing_columns)}')

df = raw_df.loc[raw_df['label'].isin(LABEL_MAP), ['text_raw', 'label']].copy()
if df.empty:
    raise ValueError('Tidak ada baris dengan label biner yang valid.')
if df['text_raw'].isna().any():
    raise ValueError(f"Ditemukan {int(df['text_raw'].isna().sum())} text_raw kosong pada subset biner. Perbaiki data sebelum training.")

# Ini hanya trim luar. Isi Unicode dan bentuk karakter teks mentah tetap dipertahankan.
df['text_raw'] = df['text_raw'].astype(str).str.strip()
if (df['text_raw'] == '').any():
    raise ValueError(f"Ditemukan {int((df['text_raw'] == '').sum())} text_raw string kosong pada subset biner.")

df['target'] = df['label'].map(LABEL_MAP)
if df['target'].isna().any():
    raise ValueError('Ada label yang gagal dipetakan.')
df['row_id'] = np.arange(len(df), dtype=int)

print('Jumlah semua workbook:', len(raw_df))
print('Jumlah subset biner:', len(df))
display(df['label'].value_counts().rename_axis('label').reset_index(name='n'))
print('Input kedua baseline: text_raw saja.')
print('Target kedua baseline: label -> target saja.')
print('Metadata TIDAK digunakan:', sorted(set(raw_df.columns) - {'text_raw', 'label'}))


## 4. Split group-aware 70/15/15 dan manifest

NFKC/casefold/whitespace dan prefix delapan token dipakai **hanya** untuk membuat grup anti-leakage. `text_raw` yang masuk model tidak dimodifikasi. Tujuh fold memberi proporsi 71.4/14.3/14.3, yang merupakan pendekatan group-aware terdekat untuk target 70/15/15.

In [ ]:
def grouping_text(text: str) -> str:
    normalized = unicodedata.normalize('NFKC', text).casefold()
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

def template_group_key(text: str) -> str:
    normalized = grouping_text(text)
    tokens = re.findall(r'\w+', normalized, flags=re.UNICODE)
    # Prefix kelompok template; bila pendek, seluruh teks membentuk key yang stabil.
    prefix = ' '.join(tokens[:8]) if tokens else normalized
    return prefix

df['normalized_group_text'] = df['text_raw'].map(grouping_text)
df['group_key'] = df['text_raw'].map(template_group_key)

n_groups = df['group_key'].nunique()
if n_groups < 7:
    raise ValueError(f'Hanya ada {n_groups} grup; minimal 7 diperlukan untuk split ini.')

# StratifiedGroupKFold menghasilkan fold tanpa group overlap. Cari pasangan fold validasi/test
# dengan deviasi distribusi kelas paling kecil dari distribusi keseluruhan.
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=RNG_SEED)
fold_id = np.empty(len(df), dtype=int)
for fold, (_, fold_idx) in enumerate(sgkf.split(df, df['target'], groups=df['group_key'])):
    fold_id[fold_idx] = fold
df['fold_id'] = fold_id

global_positive_rate = df['target'].mean()
candidates = []
for validation_fold in range(7):
    for test_fold in range(7):
        if validation_fold == test_fold:
            continue
        assignment = np.where(
            df['fold_id'].eq(test_fold), 'test',
            np.where(df['fold_id'].eq(validation_fold), 'validation', 'train'),
        )
        candidate = df.assign(split=assignment)
        parts = [candidate.loc[candidate['split'].eq(name), 'target'] for name in ('train', 'validation', 'test')]
        if any(part.nunique() != 2 for part in parts):
            continue
        rate_error = sum(abs(part.mean() - global_positive_rate) for part in parts)
        size_error = sum(abs(len(part) / len(df) - expected) for part, expected in zip(parts, (5/7, 1/7, 1/7)))
        candidates.append((rate_error + size_error, validation_fold, test_fold))

if not candidates:
    raise ValueError(
        'Tidak dapat membentuk split group-aware yang mempertahankan kedua kelas pada train/validation/test. '
        'Periksa komposisi grup template atau gunakan strategi split yang didokumentasikan ulang.'
    )

_, validation_fold, test_fold = min(candidates)
df['split'] = np.where(
    df['fold_id'].eq(test_fold), 'test',
    np.where(df['fold_id'].eq(validation_fold), 'validation', 'train'),
)

group_overlap = (
    df.groupby('group_key')['split'].nunique().gt(1).sum()
)
assert group_overlap == 0, f'Leakage: {group_overlap} group_key muncul di lebih dari satu split.'

split_summary = (
    df.groupby(['split', 'label']).size().unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .reindex(['train', 'validation', 'test'])
)
display(split_summary)
print('Fold validasi:', validation_fold, '| Fold test:', test_fold, '| Group overlap:', group_overlap)

manifest_columns = ['row_id', 'text_raw', 'label', 'target', 'normalized_group_text', 'group_key', 'fold_id', 'split']
split_manifest = df[manifest_columns].sort_values('row_id').copy()
manifest_path = OUTPUT_DIR / 'baseline_split_manifest.csv'
split_manifest.to_csv(manifest_path, index=False, encoding='utf-8')
print('Split manifest tersimpan:', manifest_path)


In [ ]:
train_df = df.loc[df['split'].eq('train'), ['row_id', 'text_raw', 'target']].reset_index(drop=True)
validation_df = df.loc[df['split'].eq('validation'), ['row_id', 'text_raw', 'target']].reset_index(drop=True)
test_df = df.loc[df['split'].eq('test'), ['row_id', 'text_raw', 'target']].reset_index(drop=True)

assert set(train_df['row_id']).isdisjoint(validation_df['row_id'])
assert set(train_df['row_id']).isdisjoint(test_df['row_id'])
assert set(validation_df['row_id']).isdisjoint(test_df['row_id'])
print(f'Train={len(train_df)}, validation={len(validation_df)}, test={len(test_df)}')


## 5. Fungsi evaluasi

Metrik utama: precision, recall, dan F1 kelas positif (`promosi_judol`), macro F1, serta PR-AUC.

In [ ]:
CLASS_NAMES = ['bukan_judol', 'promosi_judol']

def metric_dict(y_true, y_pred, positive_probability):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'positive_precision': float(precision[1]),
        'positive_recall': float(recall[1]),
        'positive_f1': float(f1[1]),
        'pr_auc': float(average_precision_score(y_true, positive_probability)),
    }
    return metrics

def save_evaluation(model_name, split_name, y_true, y_pred, positive_probability, output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics = metric_dict(y_true, y_pred, positive_probability)
    stem = f'{model_name}_{split_name}'
    (output_dir / f'{stem}_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
    (output_dir / f'{stem}_classification_report.txt').write_text(report, encoding='utf-8')

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Label sebenarnya')
    ax.set_title(f'Confusion Matrix — {model_name} ({split_name})')
    fig.tight_layout()
    fig.savefig(output_dir / f'{stem}_confusion_matrix.png', dpi=160, bbox_inches='tight')
    plt.show()

    print(f'\n=== {model_name} | {split_name} ===')
    print(json.dumps(metrics, indent=2))
    print(report)
    return metrics

## 6. Baseline klasik — TF-IDF character n-gram + Logistic Regression

Character n-gram dipilih karena lebih sensitif terhadap pola karakter daripada word n-gram dan menjadi pembanding klasik yang relevan untuk studi Unicode.

In [ ]:
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    sublinear_tf=True,
    min_df=2,
    max_features=100_000,
)

X_train_tfidf = tfidf.fit_transform(train_df['text_raw'])
X_validation_tfidf = tfidf.transform(validation_df['text_raw'])
X_test_tfidf = tfidf.transform(test_df['text_raw'])

char_model = LogisticRegression(
    max_iter=3000,
    class_weight='balanced',
    random_state=RNG_SEED,
    n_jobs=-1,
)
char_model.fit(X_train_tfidf, train_df['target'])

joblib.dump({'vectorizer': tfidf, 'classifier': char_model}, OUTPUT_DIR / 'tfidf_char_logreg.joblib')
print('Model TF-IDF + Logistic Regression disimpan.')

char_validation_prob = char_model.predict_proba(X_validation_tfidf)[:, 1]
char_validation_pred = (char_validation_prob >= 0.5).astype(int)
char_validation_metrics = save_evaluation(
    'tfidf_char_logreg', 'validation', validation_df['target'], char_validation_pred, char_validation_prob
)

char_test_prob = char_model.predict_proba(X_test_tfidf)[:, 1]
char_test_pred = (char_test_prob >= 0.5).astype(int)
char_test_metrics = save_evaluation(
    'tfidf_char_logreg', 'clean_test', test_df['target'], char_test_pred, char_test_prob
)

pd.DataFrame({
    'row_id': test_df['row_id'], 'text_raw': test_df['text_raw'], 'target': test_df['target'],
    'prediction': char_test_pred, 'positive_probability': char_test_prob,
}).to_csv(OUTPUT_DIR / 'tfidf_char_logreg_clean_test_predictions.csv', index=False, encoding='utf-8')


## 7. Baseline deep learning — fine-tuning IndoBERT

Bagian ini mengubah `text_raw` menjadi token dan melatih encoder serta classification head secara bersama-sama. Jangan menjalankan sel ini tanpa GPU bila ingin waktu training yang wajar.

In [ ]:
if not torch.cuda.is_available():
    print('PERINGATAN: GPU tidak terdeteksi. Transformer tetap dapat berjalan, tetapi akan jauh lebih lambat.')
else:
    print('GPU:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL_DIR, local_files_only=True)

def dataframe_to_dataset(frame: pd.DataFrame) -> Dataset:
    # Dataset hanya berisi text_raw dan labels. row_id dipertahankan di luar untuk artefak prediksi.
    dataset = Dataset.from_pandas(
        frame[['text_raw', 'target']].rename(columns={'target': 'labels'}),
        preserve_index=False,
    )
    return dataset.map(
        lambda batch: tokenizer(batch['text_raw'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['text_raw'],
    )

train_dataset = dataframe_to_dataset(train_df)
validation_dataset = dataframe_to_dataset(validation_df)
test_dataset = dataframe_to_dataset(test_df)

model = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED_MODEL_DIR,
    local_files_only=True,
    num_labels=2,
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_transformer_metrics(eval_prediction):
    logits, labels = eval_prediction
    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    predictions = np.argmax(logits, axis=1)
    return metric_dict(labels, predictions, probabilities)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'indobert_checkpoints'),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=RNG_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_transformer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

transformer_dir = OUTPUT_DIR / 'indobert_baseline_model'
trainer.save_model(str(transformer_dir))
tokenizer.save_pretrained(str(transformer_dir))
print('Model dan tokenizer tersimpan:', transformer_dir)


In [ ]:
transformer_validation_output = trainer.predict(validation_dataset, metric_key_prefix='validation')
transformer_validation_prob = torch.softmax(
    torch.tensor(transformer_validation_output.predictions), dim=1
).numpy()[:, 1]
transformer_validation_pred = np.argmax(transformer_validation_output.predictions, axis=1)
transformer_validation_metrics = save_evaluation(
    'indobert', 'validation', validation_df['target'], transformer_validation_pred, transformer_validation_prob
)

# Test hanya dievaluasi setelah checkpoint terbaik dipilih memakai validation set.
transformer_test_output = trainer.predict(test_dataset, metric_key_prefix='clean_test')
transformer_test_prob = torch.softmax(torch.tensor(transformer_test_output.predictions), dim=1).numpy()[:, 1]
transformer_test_pred = np.argmax(transformer_test_output.predictions, axis=1)
transformer_test_metrics = save_evaluation(
    'indobert', 'clean_test', test_df['target'], transformer_test_pred, transformer_test_prob
)

pd.DataFrame({
    'row_id': test_df['row_id'], 'text_raw': test_df['text_raw'], 'target': test_df['target'],
    'prediction': transformer_test_pred, 'positive_probability': transformer_test_prob,
}).to_csv(OUTPUT_DIR / 'indobert_clean_test_predictions.csv', index=False, encoding='utf-8')

experiment_config = {
    'dataset_path': str(DATASET_PATH), 'sheet_name': SHEET_NAME, 'input_column': 'text_raw',
    'label_column': 'label', 'label_map': LABEL_MAP, 'seed': RNG_SEED,
    'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION,
    'pretrained_model_dir': str(PRETRAINED_MODEL_DIR.resolve()),
    'max_length': MAX_LENGTH, 'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE, 'train_batch_size': TRAIN_BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE, 'split_manifest': str(manifest_path),
}
(OUTPUT_DIR / 'indobert_experiment_config.json').write_text(
    json.dumps(experiment_config, indent=2), encoding='utf-8'
)


## 8. Perbandingan clean baseline dan langkah eksperimen berikutnya

In [ ]:
comparison = pd.DataFrame([
    {'model': 'TF-IDF char (3–5) + Logistic Regression', 'split': 'validation', **char_validation_metrics},
    {'model': 'TF-IDF char (3–5) + Logistic Regression', 'split': 'clean_test', **char_test_metrics},
    {'model': 'IndoBERT fine-tuned', 'split': 'validation', **transformer_validation_metrics},
    {'model': 'IndoBERT fine-tuned', 'split': 'clean_test', **transformer_test_metrics},
])
comparison.to_csv(OUTPUT_DIR / 'baseline_comparison.csv', index=False, encoding='utf-8')
display(comparison.round(4))

print('\nArtefak penting:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)

print('''
Bagian berikutnya pada notebook ini: benchmark Unicode pada test split beku.
Benchmark tersebut memuat ulang model baseline yang sudah tersimpan, tanpa retraining.
Setelah vulnerability terukur, adversarial training berikutnya hanya boleh mengaugment train split;
validation dan test harus tetap tidak tersentuh.
''')

## 9. Benchmark robustness Unicode pada checkpoint baseline

Bagian ini **tidak melatih ulang** model dan **tidak menormalkan** input sebelum inferensi. Ia memakai `baseline_split_manifest.csv` sebagai sumber test split beku, menghasilkan satu varian per sumber untuk setiap jenis attack, lalu mengevaluasi checkpoint TF-IDF dan IndoBERT yang sudah disimpan.

Tiga attack intensitas sedang (`30%`) diuji secara terpisah agar efeknya dapat diatribusikan:

1. `cross_script_confusable_substitution` — substitusi Latin ke karakter Yunani/Sirilic yang tampak serupa;
2. `zero_width_insertion` — penyisipan U+200B pada batas internal alfanumerik;
3. `combining_mark_insertion` — penyisipan satu atau beberapa *combining marks* (Zalgo-like/diacritic overlay) setelah karakter dasar Latin, seperti contoh `h̟̜̆̔a͚ͩl͕͉ͣ͋͘o`.

> *Cross-script confusable* dan *combining-mark insertion* adalah dua keluarga attack Unicode yang berbeda. Benchmark ini mengukurnya secara terpisah; hasil macro-average hanya ringkasan tambahan, bukan pengganti laporan per attack.

> Untuk menjalankan benchmark saja setelah training selesai, jalankan sel **1–5**, sel **fungsi evaluasi (§5)**, lalu sel **§9–§11**. Sel load dataset, split, dan training (§3, §4, §6, §7) tidak perlu dijalankan. Atur `BASELINE_ARTIFACT_DIR` ke folder output baseline yang telah disalin; gunakan `BENCHMARK_OUTPUT_DIR` yang dapat ditulis untuk menyimpan hasil baru.

In [ ]:
# Gunakan OUTPUT_DIR bila training dan benchmark dilakukan dalam runtime/folder yang sama.
# Contoh Drive: BASELINE_ARTIFACT_DIR = Path('/content/drive/MyDrive/judol_detection/judol_baseline_outputs')
# Contoh Kaggle: BASELINE_ARTIFACT_DIR = Path('/kaggle/input/NAMA-ARTEFAK/judol_baseline_outputs')
BASELINE_ARTIFACT_DIR = OUTPUT_DIR  # <-- UBAH BILA ARTEFAK BASELINE BERADA DI LOKASI LAIN
# Gunakan folder baru agar hasil benchmark v2 tidak menimpa benchmark awal dua attack.
BENCHMARK_OUTPUT_DIR = OUTPUT_DIR / 'unicode_benchmark_v2'

ATTACK_SEED = 42
ATTACK_INTENSITY = 0.30  # Proporsi karakter dasar eligible yang dipilih per attack.
ATTACK_TYPES = (
    'cross_script_confusable_substitution',
    'zero_width_insertion',
    'combining_mark_insertion',
)
COMBINING_MARKS_PER_SELECTED_BASE = (1, 4)
ATTACK_GENERATOR_VERSION = 'unicode_benchmark_v2'
INFERENCE_BATCH_SIZE = 32

if not 0 < ATTACK_INTENSITY <= 1:
    raise ValueError('ATTACK_INTENSITY harus berada pada rentang (0, 1].')
if not ATTACK_TYPES:
    raise ValueError('ATTACK_TYPES tidak boleh kosong.')
if (
    len(COMBINING_MARKS_PER_SELECTED_BASE) != 2
    or COMBINING_MARKS_PER_SELECTED_BASE[0] < 1
    or COMBINING_MARKS_PER_SELECTED_BASE[0] > COMBINING_MARKS_PER_SELECTED_BASE[1]
):
    raise ValueError('COMBINING_MARKS_PER_SELECTED_BASE harus berupa tuple (minimum, maksimum) dengan minimum >= 1.')

BENCHMARK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
required_paths = {
    'split manifest': BASELINE_ARTIFACT_DIR / 'baseline_split_manifest.csv',
    'TF-IDF model': BASELINE_ARTIFACT_DIR / 'tfidf_char_logreg.joblib',
    'IndoBERT model': BASELINE_ARTIFACT_DIR / 'indobert_baseline_model',
    'TF-IDF clean metrics': BASELINE_ARTIFACT_DIR / 'tfidf_char_logreg_clean_test_metrics.json',
    'IndoBERT clean metrics': BASELINE_ARTIFACT_DIR / 'indobert_clean_test_metrics.json',
}
missing_artifacts = [f'{name}: {path}' for name, path in required_paths.items() if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        'Artefak baseline berikut tidak ditemukan. Salin seluruh folder output baseline atau ubah '
        'BASELINE_ARTIFACT_DIR:\n- ' + '\n- '.join(missing_artifacts)
    )

frozen_manifest = pd.read_csv(required_paths['split manifest'])
required_manifest_columns = {'row_id', 'text_raw', 'target', 'split'}
missing_manifest_columns = required_manifest_columns - set(frozen_manifest.columns)
if missing_manifest_columns:
    raise ValueError(f'Kolom manifest wajib tidak ditemukan: {sorted(missing_manifest_columns)}')

frozen_test_df = frozen_manifest.loc[
    frozen_manifest['split'].eq('test'), ['row_id', 'text_raw', 'target', 'split']
].copy().sort_values('row_id').reset_index(drop=True)
if frozen_test_df.empty:
    raise ValueError('Tidak ada baris test pada baseline_split_manifest.csv.')
if frozen_test_df['row_id'].duplicated().any():
    raise ValueError('row_id pada test split manifest tidak unik.')
if set(frozen_test_df['target'].unique()) != {0, 1}:
    raise ValueError('Test split beku harus memuat kedua kelas 0 dan 1.')

print('Baseline artifacts :', BASELINE_ARTIFACT_DIR.resolve())
print('Benchmark output  :', BENCHMARK_OUTPUT_DIR.resolve())
print('Generator version :', ATTACK_GENERATOR_VERSION)
print('Jumlah test beku  :', len(frozen_test_df))
display(frozen_test_df['target'].value_counts().rename_axis('target').reset_index(name='n'))

In [ ]:
# Latin -> karakter lintas-skrip yang secara visual mirip.
# Mapping eksplisit ini adalah attack cross-script confusable terkontrol, bukan tabel UTS #39 lengkap.
CROSS_SCRIPT_CONFUSABLE_MAP = {
    'a': 'а', 'c': 'с', 'e': 'е', 'i': 'і', 'j': 'ј', 'o': 'ο', 'p': 'р', 's': 'ѕ', 'x': 'х', 'y': 'у',
    'A': 'А', 'B': 'Β', 'C': 'С', 'E': 'Е', 'H': 'Н', 'I': 'І', 'K': 'Κ', 'M': 'М', 'O': 'Ο', 'P': 'Р',
    'T': 'Т', 'X': 'Χ', 'Y': 'Υ',
}
ZERO_WIDTH_CHARACTER = '​'  # ZERO WIDTH SPACE

# Pool eksplisit combining nonspacing marks (Mn): atas, bawah, dan overlay.
# Pool ini merepresentasikan combining-mark/Zalgo-like attack secara terkontrol,
# bukan enumerasi semua karakter diakritik Unicode.
COMBINING_MARK_POOL = (
    '̀', '́', '̂', '̃', '̄', '̆', '̇', '̈',
    '̊', '̋', '̌', '̑', '̓', '̔', '̕', '̚',
    '̛', '̣', '̤', '̥', '̦', '̧', '̨', '̭',
    '̮', '̰', '̱', '̲', '̴', '̵', '̶', '̷',
    '̸', '͂', '͆', '͊', '͏', '͒', '͗', '͜',
    '͠', '͡', '͢',
)
assert all(unicodedata.category(mark) == 'Mn' for mark in COMBINING_MARK_POOL)


def stable_attack_seed(row_id: int, attack_type: str) -> int:
    payload = f'{ATTACK_SEED}|{int(row_id)}|{attack_type}'.encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], byteorder='big')


def n_to_modify(candidate_count: int) -> int:
    if candidate_count == 0:
        return 0
    return max(1, int(np.ceil(candidate_count * ATTACK_INTENSITY)))


def apply_cross_script_confusable_substitution(text: str, rng: np.random.Generator) -> tuple[str, int, int, int]:
    candidates = [index for index, character in enumerate(text) if character in CROSS_SCRIPT_CONFUSABLE_MAP]
    selected_count = n_to_modify(len(candidates))
    selected = set(rng.choice(candidates, size=selected_count, replace=False).tolist()) if selected_count else set()
    attacked_text = ''.join(
        CROSS_SCRIPT_CONFUSABLE_MAP[character] if index in selected else character
        for index, character in enumerate(text)
    )
    return attacked_text, len(candidates), selected_count, 0


def apply_zero_width_insertion(text: str, rng: np.random.Generator) -> tuple[str, int, int, int]:
    # Kandidat adalah batas di dalam token alfanumerik, mis. "slot" -> "sl​ot".
    boundaries = [index for index in range(1, len(text)) if text[index - 1].isalnum() and text[index].isalnum()]
    selected_count = n_to_modify(len(boundaries))
    selected = set(rng.choice(boundaries, size=selected_count, replace=False).tolist()) if selected_count else set()
    pieces = []
    for index, character in enumerate(text):
        if index in selected:
            pieces.append(ZERO_WIDTH_CHARACTER)
        pieces.append(character)
    return ''.join(pieces), len(boundaries), selected_count, 0


def is_latin_letter_base(character: str) -> bool:
    """Only ordinary Latin base letters are eligible for combining-mark insertion."""
    return (
        len(character) == 1
        and ('A' <= character <= 'Z' or 'a' <= character <= 'z')
        and unicodedata.combining(character) == 0
    )


def apply_combining_mark_insertion(text: str, rng: np.random.Generator) -> tuple[str, int, int, int]:
    """Attach a deterministic bounded stack of Mn marks after selected Latin bases.

    ATTACK_INTENSITY controls the fraction of eligible base letters selected;
    COMBINING_MARKS_PER_SELECTED_BASE independently controls stack depth.
    Existing source code points are retained verbatim.
    """
    candidates = [index for index, character in enumerate(text) if is_latin_letter_base(character)]
    selected_count = n_to_modify(len(candidates))
    selected = set(rng.choice(candidates, size=selected_count, replace=False).tolist()) if selected_count else set()
    min_marks, max_marks = COMBINING_MARKS_PER_SELECTED_BASE
    pieces = []
    inserted_mark_count = 0
    for index, character in enumerate(text):
        pieces.append(character)
        if index in selected:
            mark_count = int(rng.integers(min_marks, max_marks + 1))
            marks = rng.choice(COMBINING_MARK_POOL, size=mark_count, replace=True).tolist()
            pieces.extend(marks)
            inserted_mark_count += mark_count
    return ''.join(pieces), len(candidates), selected_count, inserted_mark_count


def build_attack_row(source_row: pd.Series, attack_type: str) -> dict:
    row_id = int(source_row['row_id'])
    source_text = str(source_row['text_raw'])
    attack_seed = stable_attack_seed(row_id, attack_type)
    rng = np.random.default_rng(attack_seed)

    if attack_type == 'cross_script_confusable_substitution':
        adversarial_text, eligible_count, modified_count, inserted_mark_count = apply_cross_script_confusable_substitution(source_text, rng)
    elif attack_type == 'zero_width_insertion':
        adversarial_text, eligible_count, modified_count, inserted_mark_count = apply_zero_width_insertion(source_text, rng)
    elif attack_type == 'combining_mark_insertion':
        adversarial_text, eligible_count, modified_count, inserted_mark_count = apply_combining_mark_insertion(source_text, rng)
    else:
        raise ValueError(f'Attack type tidak didukung: {attack_type}')

    return {
        'source_row_id': row_id,
        'split': source_row['split'],
        'target': int(source_row['target']),
        'attack_type': attack_type,
        'attack_intensity': ATTACK_INTENSITY,
        'combining_marks_per_selected_base': (
            f'{COMBINING_MARKS_PER_SELECTED_BASE[0]}-{COMBINING_MARKS_PER_SELECTED_BASE[1]}'
            if attack_type == 'combining_mark_insertion' else None
        ),
        'attack_seed': attack_seed,
        'attack_generator_version': ATTACK_GENERATOR_VERSION,
        'source_text_raw': source_text,
        'adversarial_text_raw': adversarial_text,
        'eligible_count': eligible_count,
        'modified_count': modified_count,
        'inserted_mark_count': inserted_mark_count,
        'attack_applied': bool(modified_count > 0),
    }


def build_attack_manifest() -> pd.DataFrame:
    attack_records = [
        build_attack_row(source_row, attack_type)
        for attack_type in ATTACK_TYPES
        for _, source_row in frozen_test_df.iterrows()
    ]
    return pd.DataFrame(attack_records).sort_values(
        ['attack_type', 'source_row_id']
    ).reset_index(drop=True)


def non_mark_codepoint_sequence(text: str) -> str:
    return ''.join(character for character in text if unicodedata.category(character) != 'Mn')


def count_combining_marks(text: str) -> int:
    return sum(unicodedata.category(character) == 'Mn' for character in text)


def validate_combining_mark_row(source_text: str, adversarial_text: str) -> int:
    """Verify only Mn code points were added; source non-mark code points stay ordered."""
    if non_mark_codepoint_sequence(source_text) != non_mark_codepoint_sequence(adversarial_text):
        raise AssertionError('Urutan code point non-mark berubah pada combining-mark attack.')
    inserted_count = count_combining_marks(adversarial_text) - count_combining_marks(source_text)
    if inserted_count < 0:
        raise AssertionError('Jumlah combining marks tidak boleh berkurang.')
    return inserted_count


adversarial_test_manifest = build_attack_manifest()
# Dua pembuatan manifest dengan konfigurasi sama harus identik pada seluruh nilai tabel.
pd.testing.assert_frame_equal(adversarial_test_manifest, build_attack_manifest())

expected_rows = len(frozen_test_df) * len(ATTACK_TYPES)
assert len(adversarial_test_manifest) == expected_rows
assert set(adversarial_test_manifest['attack_type']) == set(ATTACK_TYPES)
assert adversarial_test_manifest.groupby('attack_type')['source_row_id'].nunique().eq(len(frozen_test_df)).all()
assert adversarial_test_manifest[['attack_type', 'source_row_id']].duplicated().sum() == 0
assert adversarial_test_manifest['split'].eq('test').all()
assert set(adversarial_test_manifest['source_row_id']) == set(frozen_test_df['row_id'])
source_target_by_id = frozen_test_df.set_index('row_id')['target']
assert adversarial_test_manifest['target'].eq(
    adversarial_test_manifest['source_row_id'].map(source_target_by_id)
).all()

combining_rows = adversarial_test_manifest.loc[
    adversarial_test_manifest['attack_type'].eq('combining_mark_insertion')
].copy()
actual_inserted_counts = combining_rows.apply(
    lambda row: validate_combining_mark_row(row['source_text_raw'], row['adversarial_text_raw']),
    axis=1,
)
assert actual_inserted_counts.eq(combining_rows['inserted_mark_count']).all()
assert combining_rows['modified_count'].eq(
    combining_rows['eligible_count'].map(n_to_modify)
).where(combining_rows['eligible_count'].gt(0), combining_rows['modified_count'].eq(0)).all()
assert combining_rows.loc[combining_rows['attack_applied'], 'inserted_mark_count'].between(
    combining_rows.loc[combining_rows['attack_applied'], 'modified_count'] * COMBINING_MARKS_PER_SELECTED_BASE[0],
    combining_rows.loc[combining_rows['attack_applied'], 'modified_count'] * COMBINING_MARKS_PER_SELECTED_BASE[1],
).all()
assert combining_rows.loc[~combining_rows['attack_applied'], 'inserted_mark_count'].eq(0).all()

attack_manifest_path = BENCHMARK_OUTPUT_DIR / 'unicode_adversarial_test_manifest.csv'
adversarial_test_manifest.to_csv(attack_manifest_path, index=False, encoding='utf-8')
print('Manifest adversarial tersimpan:', attack_manifest_path)

attack_coverage = (
    adversarial_test_manifest.groupby(['attack_type', 'target'])
    .agg(
        n=('source_row_id', 'size'),
        applied=('attack_applied', 'sum'),
        no_change=('attack_applied', lambda values: int((~values).sum())),
        mean_eligible=('eligible_count', 'mean'),
        mean_modified=('modified_count', 'mean'),
        total_inserted_combining_marks=('inserted_mark_count', 'sum'),
        mean_inserted_combining_marks=('inserted_mark_count', 'mean'),
    )
    .reset_index()
)
attack_coverage['applied_rate'] = attack_coverage['applied'] / attack_coverage['n']
display(attack_coverage.round(3))

examples = adversarial_test_manifest.loc[
    adversarial_test_manifest['attack_applied']
].groupby('attack_type', group_keys=False).head(2).copy()
examples['unicode_codepoints'] = examples['adversarial_text_raw'].map(
    lambda text: ' '.join(f'U+{ord(character):04X}' for character in text[:120])
)
display(examples[[
    'attack_type', 'source_row_id', 'source_text_raw', 'adversarial_text_raw',
    'modified_count', 'inserted_mark_count', 'unicode_codepoints',
]])

## 10. Inferensi checkpoint baseline pada test Unicode

Checkpoint, tokenizer, dan vectorizer dimuat ulang dari artefak baseline. Tidak ada pemanggilan `fit`, `train`, normalizer, atau perubahan threshold pada bagian ini.

In [ ]:
def predict_indobert_texts(model, tokenizer, texts: list[str]) -> np.ndarray:
    model.eval()
    device = next(model.parameters()).device
    probabilities = []
    with torch.no_grad():
        for start_index in range(0, len(texts), INFERENCE_BATCH_SIZE):
            batch_texts = texts[start_index:start_index + INFERENCE_BATCH_SIZE]
            encoded = tokenizer(
                batch_texts,
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors='pt',
            )
            encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
            logits = model(**encoded).logits
            probabilities.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy().tolist())
    return np.asarray(probabilities, dtype=float)

saved_char_bundle = joblib.load(required_paths['TF-IDF model'])
if not {'vectorizer', 'classifier'} <= set(saved_char_bundle):
    raise ValueError('tfidf_char_logreg.joblib harus berisi key vectorizer dan classifier.')
saved_tfidf = saved_char_bundle['vectorizer']
saved_char_model = saved_char_bundle['classifier']

saved_tokenizer = AutoTokenizer.from_pretrained(required_paths['IndoBERT model'])
saved_indobert = AutoModelForSequenceClassification.from_pretrained(required_paths['IndoBERT model'])
inference_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
saved_indobert.to(inference_device)
saved_indobert.eval()

print('TF-IDF checkpoint dimuat:', required_paths['TF-IDF model'])
print('IndoBERT checkpoint dimuat:', required_paths['IndoBERT model'])
print('Device inferensi IndoBERT:', inference_device)

unicode_metrics_records = []
for attack_type in ATTACK_TYPES:
    attack_df = adversarial_test_manifest.loc[
        adversarial_test_manifest['attack_type'].eq(attack_type)
    ].reset_index(drop=True)
    y_true = attack_df['target'].to_numpy(dtype=int)
    texts = attack_df['adversarial_text_raw'].tolist()

    char_probabilities = saved_char_model.predict_proba(saved_tfidf.transform(texts))[:, 1]
    char_predictions = (char_probabilities >= 0.5).astype(int)
    char_metrics = save_evaluation(
        'tfidf_char_logreg', f'unicode_{attack_type}', y_true, char_predictions,
        char_probabilities, output_dir=BENCHMARK_OUTPUT_DIR,
    )
    pd.DataFrame({
        'source_row_id': attack_df['source_row_id'], 'attack_type': attack_type,
        'target': y_true, 'attack_applied': attack_df['attack_applied'],
        'prediction': char_predictions, 'positive_probability': char_probabilities,
    }).to_csv(
        BENCHMARK_OUTPUT_DIR / f'tfidf_char_logreg_unicode_{attack_type}_predictions.csv',
        index=False, encoding='utf-8',
    )
    unicode_metrics_records.append({
        'model': 'TF-IDF char (3–5) + Logistic Regression',
        'model_key': 'tfidf_char_logreg', 'attack_type': attack_type, **char_metrics,
    })

    indobert_probabilities = predict_indobert_texts(saved_indobert, saved_tokenizer, texts)
    indobert_predictions = (indobert_probabilities >= 0.5).astype(int)
    indobert_metrics = save_evaluation(
        'indobert', f'unicode_{attack_type}', y_true, indobert_predictions,
        indobert_probabilities, output_dir=BENCHMARK_OUTPUT_DIR,
    )
    pd.DataFrame({
        'source_row_id': attack_df['source_row_id'], 'attack_type': attack_type,
        'target': y_true, 'attack_applied': attack_df['attack_applied'],
        'prediction': indobert_predictions, 'positive_probability': indobert_probabilities,
    }).to_csv(
        BENCHMARK_OUTPUT_DIR / f'indobert_unicode_{attack_type}_predictions.csv',
        index=False, encoding='utf-8',
    )
    unicode_metrics_records.append({
        'model': 'IndoBERT fine-tuned',
        'model_key': 'indobert', 'attack_type': attack_type, **indobert_metrics,
    })

unicode_attack_metrics = pd.DataFrame(unicode_metrics_records)
unicode_attack_metrics.to_csv(
    BENCHMARK_OUTPUT_DIR / 'unicode_attack_metrics.csv', index=False, encoding='utf-8'
)
display(unicode_attack_metrics.round(4))

## 11. Perbandingan paired clean vs Unicode dan robustness drop

`robustness_drop = metrik_clean_test − metrik_adversarial`. Nilai positif berarti performa turun akibat attack. Ringkasan agregat adalah macro-average atas **tiga keluarga attack yang dikonfigurasi**; bukan skor campuran clean dan adversarial. Hasil per attack tetap menjadi hasil utama.

Untuk eksperimen adversarial training berikutnya, keluarga attack pada augmentasi train harus dideklarasikan sebelum retraining. Sedikitnya satu keluarga Unicode harus tidak pernah dipakai untuk augmentasi train maupun pemilihan hyperparameter, lalu hanya dievaluasi pada test split beku sebagai manipulasi Unicode *unseen/zero-day*. `combining_mark_insertion` adalah kandidat keluarga held-out tersebut; keputusan final harus ditetapkan sebelum defense dilatih.

In [ ]:
METRIC_COLUMNS = [
    'accuracy', 'macro_f1', 'positive_precision', 'positive_recall', 'positive_f1', 'pr_auc'
]

clean_metric_paths = {
    'tfidf_char_logreg': required_paths['TF-IDF clean metrics'],
    'indobert': required_paths['IndoBERT clean metrics'],
}
clean_metrics_records = []
for model_key, metric_path in clean_metric_paths.items():
    clean_metrics = json.loads(metric_path.read_text(encoding='utf-8'))
    missing_metrics = set(METRIC_COLUMNS) - set(clean_metrics)
    if missing_metrics:
        raise ValueError(f'Metrik clean tidak lengkap pada {metric_path}: {sorted(missing_metrics)}')
    clean_metrics_records.append({'model_key': model_key, 'scenario': 'clean_test', **clean_metrics})
clean_metrics_df = pd.DataFrame(clean_metrics_records)

robustness_comparison = unicode_attack_metrics.merge(
    clean_metrics_df,
    on='model_key',
    how='inner',
    suffixes=('_adversarial', '_clean'),
    validate='many_to_one',
)
if len(robustness_comparison) != len(unicode_attack_metrics):
    raise AssertionError('Sebagian hasil attack tidak memiliki metrik clean pasangan.')
for metric_name in METRIC_COLUMNS:
    robustness_comparison[f'{metric_name}_drop'] = (
        robustness_comparison[f'{metric_name}_clean'] - robustness_comparison[f'{metric_name}_adversarial']
    )

comparison_columns = [
    'model', 'model_key', 'attack_type',
    *[f'{metric_name}_clean' for metric_name in METRIC_COLUMNS],
    *[f'{metric_name}_adversarial' for metric_name in METRIC_COLUMNS],
    *[f'{metric_name}_drop' for metric_name in METRIC_COLUMNS],
]
robustness_comparison = robustness_comparison[comparison_columns].sort_values(['model_key', 'attack_type'])
robustness_comparison.to_csv(
    BENCHMARK_OUTPUT_DIR / 'unicode_robustness_comparison.csv', index=False, encoding='utf-8'
)

aggregate_rows = []
for (model, model_key), group in robustness_comparison.groupby(['model', 'model_key'], sort=True):
    observed_attacks = set(group['attack_type'])
    if observed_attacks != set(ATTACK_TYPES):
        raise AssertionError(
            f'Attack tidak lengkap untuk {model_key}: expected={sorted(ATTACK_TYPES)}, observed={sorted(observed_attacks)}'
        )
    aggregate_row = {
        'model': model,
        'model_key': model_key,
        'attack_type': 'macro_average_across_attack_types',
        'n_attack_types': int(group['attack_type'].nunique()),
        'attack_generator_version': ATTACK_GENERATOR_VERSION,
    }
    for metric_name in METRIC_COLUMNS:
        aggregate_row[f'{metric_name}_clean'] = float(group[f'{metric_name}_clean'].iloc[0])
        aggregate_row[f'{metric_name}_adversarial'] = float(group[f'{metric_name}_adversarial'].mean())
        aggregate_row[f'{metric_name}_drop'] = float(group[f'{metric_name}_drop'].mean())
    aggregate_rows.append(aggregate_row)
robustness_summary = pd.DataFrame(aggregate_rows)
robustness_summary.to_csv(
    BENCHMARK_OUTPUT_DIR / 'unicode_robustness_summary.csv', index=False, encoding='utf-8'
)

display(robustness_comparison.round(4))
print(f'\nMacro-average antar {len(ATTACK_TYPES)} attack type:')
display(robustness_summary.round(4))

print('''
Interpretasi:
- Robustness drop positif berarti metrik clean lebih tinggi daripada metrik pada attack.
- Laporkan hasil per attack type dan macro-average; jangan hanya menggunakan agregat.
- Tahap berikutnya: deklarasikan attack known untuk adversarial training pada train split saja.
  Sisakan sedikitnya satu keluarga attack yang tidak pernah dipakai saat training/tuning sebagai
  evaluasi Unicode unseen/zero-day pada test split beku. Validation dan test tetap tidak tersentuh.
''')